In [1]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

/home/dgclasher/Development/projects/ai/langchain_proj/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def load_pdfs(file_path: str = "./files") -> list[Document]:
    documents = []
    for pdf_file in Path(file_path).glob("*.pdf"):
        docs = PyPDFLoader(str(pdf_file)).load()
        for doc in docs:
            doc.metadata.update({
                "source": pdf_file.name,
                "file_type": "pdf"
            })
        documents.extend(docs)
    return documents

docs = load_pdfs()

In [4]:
def split_documents(documents: list[Document], chunk_size: int = 500, chunk_overlap: int = 50):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks.")

    return split_docs

In [5]:
chunks = split_documents(docs)

Split 12 documents into 51 chunks.


In [6]:
import os
from dotenv import load_dotenv
load_dotenv()
from huggingface_hub import login
HF_TOKEN = os.environ.get("HF_TOKEN")
if HF_TOKEN:
    login(HF_TOKEN)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [7]:
import uuid
import chromadb
import numpy as np
from chromadb.config import Settings
from typing import List, Dict, Any, Tuple
from sentence_transformers import SentenceTransformer

class EmbeddingManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.device = "cpu"
        self.model = None
        self._load_model()

    def _load_model(self):
        try:
            self.model = SentenceTransformer(self.model_name, device=self.device)
            print(f"Loaded embedding model: {self.model_name} on {self.device}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        if not self.model:
            raise ValueError("Model is not loaded")
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, convert_to_numpy=True, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

embedding_manager = EmbeddingManager()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 127.98it/s]


Loaded embedding model: all-MiniLM-L6-v2 on cpu


In [8]:
class VectorStore:
    def __init__(self, collection_name: str = "policy_docs", persist_directory: str = "./vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_chromadb()

    def _initialize_chromadb(self):
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            self.collection = self.client.get_or_create_collection(name=self.collection_name)
            print(f"Initialized ChromaDB collection: {self.collection_name} at {self.persist_directory}")
        except Exception as e:
            print(f"Error initializing ChromaDB: {e}")
            raise

    def add_documents(self, documents: List[Document], embeddings: np.ndarray):
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents and embeddings must match")
        print(f"Adding {len(documents)} documents to the vector store...")
        ids = []
        metadatas = []
        document_texts = []
        embedding_list = []

        for i, (doc, emb) in  enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            document_texts.append(doc.page_content)
            embedding_list.append(emb.tolist())

        try:
            self.collection.add(
                ids=ids,
                metadatas=metadatas,
                documents=document_texts,
                embeddings=embedding_list
            )
        except Exception as e:
            print(f"Error adding documents to ChromaDB: {e}")
            raise

        print(f"Successfully added {len(documents)} documents to the vector store.")

vector_store = VectorStore()


Initialized ChromaDB collection: policy_docs at ./vector_store


In [9]:
texts = [doc.page_content for doc in chunks]
embeddings = embedding_manager.generate_embeddings(texts)
vector_store.add_documents(chunks, embeddings)

Generating embeddings for 51 texts...


Batches: 100%|██████████| 2/2 [00:08<00:00,  4.42s/it]


Generated embeddings with shape: (51, 384)
Adding 51 documents to the vector store...
Successfully added 51 documents to the vector store.


In [10]:
class RAGRetriever:
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5) -> List[Dict[str, Any]]:
        if not query:
            raise ValueError("Query must not be empty")
        print(f"Generating query embedding for: {query}")
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        print(f"Retrieving top {top_k} documents for the query...")
        results = self.vector_store.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=top_k,
            include=["documents", "metadatas", "distances"]
        )
        retrieved_docs = []
        if results and results["documents"] and results["documents"][0]:
            for i in range(len(results["documents"][0])):
                content = results["documents"][0][i]
                metadata = results["metadatas"][0][i]
                distance = results["distances"][0][i]
                retrieved_docs.append({
                    "content": content,
                    "metadata": metadata,
                    "distance": distance
                })
        else:
            print("No documents retrieved for the query.")
        return retrieved_docs
rag_retriever = RAGRetriever(vector_store, embedding_manager)

In [11]:
retrieved = rag_retriever.retrieve("What are the core principles of Nexora's Employee Code of Conduct?")
retrieved

Generating query embedding for: What are the core principles of Nexora's Employee Code of Conduct?
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.09it/s]


Generated embeddings with shape: (1, 384)
Retrieving top 5 documents for the query...


[{'content': '10\nAcknowledgment\n1. Purpose and Principles\nNexora Technologies expects every person acting on its behalf to work with integrity, professionalism and respect.\nThis policy sets baseline behavioral expectations across offices, customer sites, remote work and\ncompany-sponsored events.\nCore principles: Act honestly. Treat people fairly. Protect company and customer information. Take ownership. Speak up\nearly when something may be wrong.',
  'metadata': {'title': '(anonymous)',
   'content_length': 436,
   'total_pages': 4,
   'creationdate': '2026-10-02T14:53:03+00:00',
   'keywords': '',
   'moddate': '2026-10-02T14:53:03+00:00',
   'page': 1,
   'creator': '(unspecified)',
   'doc_index': 37,
   'source': 'Nexora_Employee_Code_of_Conduct.pdf',
   'trapped': '/False',
   'subject': '(unspecified)',
   'page_label': '2',
   'author': '(anonymous)',
   'file_type': 'pdf',
   'producer': 'ReportLab PDF Library - (opensource)'},
  'distance': 0.5842620134353638},
 {'conte

In [12]:
import litellm

litellm.callbacks = []
litellm.success_callback = []
litellm.failure_callback = []
litellm._async_success_callback = []
litellm._async_failure_callback = []
litellm.cache = None

print("LiteLLM callbacks and cache have been reset.")

LiteLLM callbacks and cache have been reset.


In [21]:
import time
from litellm import completion
from litellm.caching import Cache

litellm.cache = Cache(type="local")

start = time.time()

system_prompt = """
You are a helpful assistant that provides information based on the retrieved documents.
You will be given a user query and a set of retrieved documents. Your task is to generate a concise and accurate response to the user's query using the information from the retrieved documents.
If the information is not available in the retrieved documents, respond with "I'm sorry, I don't have that information."""

def chatbot(query: str) -> str:
    retrieved_docs = rag_retriever.retrieve(query)

    context = "\n\n".join([f"Document {i+1}:\n{doc['content']}" for i, doc in enumerate(retrieved_docs)])
    prompt = f"User Query: {query}\n\nRetrieved Documents:\n{context}\n\nPlease provide a concise and accurate response to the user's query based on the retrieved documents."

    response = completion(
        model="groq/qwen/qwen3.8-27b",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt}
        ],
        fallbacks = [
            "openrouter/deepseek/deepseek-v4.1-flash"
        ]
    )
    return response.choices[0].message

In [22]:
query = "What are the core principles of Nexora's Employee Code of Conduct?"
response = chatbot(query)

Generating query embedding for: What are the core principles of Nexora's Employee Code of Conduct?
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  9.48it/s]

Generated embeddings with shape: (1, 384)
Retrieving top 5 documents for the query...



21:23:45 - LiteLLM:WARNING: logging_worker.py:129 - LoggingWorker: event loop changed; carried 3 pending and revived 0 dequeued logging task(s) onto the new loop
Task was destroyed but it is pending!
task: <Task pending name='Task-823' coro=<LoggingWorker._worker_loop() running at /home/dgclasher/Development/projects/ai/langchain_proj/.venv/lib/python3.12/site-packages/litellm/litellm_core_utils/logging_worker.py:174>>


In [23]:
print(response.content)

The core principles of Nexora's Employee Code of Conduct are:

*   Act honestly.
*   Treat people fairly.
*   Protect company and customer information.
*   Take ownership.
*   Speak up early when something may be wrong.
